# RLAIF AI Preference Data Generator

In [1]:
import os, json, math, random
from typing import Dict
from tqdm import tqdm
from datasets import load_dataset
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\laras\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------------------- Settings  --------------------
dataset_name   = "nvidia/HelpSteer2"
data_dir       = "preference"
split          = "train"

feedback_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
principles_path= "mock_principles.json"

seed           = 123
max_examples   = 500

# -------------------------------------------------------------

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
random.seed(seed)

In [ ]:
def load_principles(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list) or not all(isinstance(x, str) for x in data):
        raise ValueError("Principles file must be a JSON array of strings.")
    return data

def feedback_prompt(principle, prompt, resp1, resp2, use_principles: bool = True):
    base = (
        "Consider the following conversation between a human and an assistant.\n"
        "[HUMAN/ASSISTANT CONVERSATION]\n"
        f"[HUMAN] {prompt}\n"
        f"[ASSISTANT A] {resp1}\n"
        f"[ASSISTANT B] {resp2}\n"
    )
    if use_principles:
        base += (
            "[PRINCIPLE FOR MULTIPLE CHOICE EVALUATION]\n"
            f"{principle}\n"
        )
    base += (
        "Options:\n"
        "  (A) [RESPONSE A]\n"
        "  (B) [RESPONSE B]\n"
        "The answer is:"
    )
    return base

def first_token_logprobs(model, tok, prefix: str) -> Dict[str, float]:
    with torch.no_grad():
        enc = tok(prefix, return_tensors="pt").to(model.device)
        out = model(**enc)
        logits = out.logits[0, -1]
        logprobs = torch.log_softmax(logits, dim=-1)
    def best(choice):
        cands = [choice, f" {choice}", f"({choice})", f"{choice})", f"[{choice}]"]
        vals = []
        for c in cands:
            ids = tok.encode(c, add_special_tokens=False)
            if len(ids) == 1:
                vals.append(float(logprobs[ids[0]].item()))
        if not vals:
            ids = tok.encode(cands[0], add_special_tokens=False)
            vals.append(float(logprobs[ids[0]].item()))
        return max(vals)
    return {"A": best("A"), "B": best("B")}

def normalize_probs(lp: Dict[str, float]) -> Dict[str, float]:
    m = max(lp.values())
    ea, eb = math.exp(lp["A"] - m), math.exp(lp["B"] - m)
    z = ea + eb
    return {"A": ea / z, "B": eb / z}

def generate_dataset(use_principles,out_json):
    # Dataset
    ds = load_dataset(dataset_name, data_dir=data_dir, split=split)
    ds = ds.shuffle(seed=seed).select(range(min(max_examples, len(ds))))
    print(f"[data] Subset size: {len(ds)}")

    # Principles 
    if use_principles:
        principles = load_principles(principles_path)
        print(f"[principles] Loaded: {len(principles)}")
    else:
        principles = [""]  
        print("[principles] Skipped (use_principles=False)")

    # Model
    device_map = "auto" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32

    tok = AutoTokenizer.from_pretrained(feedback_model, use_fast=True)
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(feedback_model, torch_dtype=dtype, device_map=device_map)
    print("[model] Loaded on", "cuda" if torch.cuda.is_available() else "cpu")

    records = []
    for row in tqdm(ds, total=len(ds), desc="Labeling"):
        prompt = row["prompt"]
        resp1 = row["response_1"]
        resp2 = row["response_2"]
        pr     = random.choice(principles)

        fb = feedback_prompt(principle=pr, prompt=prompt, resp1=resp1, resp2=resp2, use_principles=use_principles)
        lp = first_token_logprobs(model, tok, fb)
        probs = normalize_probs(lp)
        label = "A" if probs["A"] >= probs["B"] else "B"

        records.append({
            "prompt": prompt,
            "A": resp1, "B": resp2,
            "targets": {"A": probs["A"], "B": probs["B"]},
            "label": label,
            "principle": pr if use_principles else "",
            "feedback_model": feedback_model,
            "helpsteer_meta": {
                "preference_strength": row.get("preference_strength", None),
                "preference_statement": row.get("preference_statement", None)
            }
        })

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    print(f"[done] Saved {len(records)} → {out_json}")

In [ ]:
# With principles
use_principles = True
out_json = f"ai_feedback_preferences_with_principles{max_examples}.json"
generate_dataset(use_principles,out_json)


[data] Subset size: 500
[principles] Loaded: 3


`torch_dtype` is deprecated! Use `dtype` instead!


[model] Loaded on cpu


Labeling: 100%|████████████████████████████████████████████████████████████████████| 500/500 [2:16:27<00:00, 16.38s/it]


[done] Saved 500 → ai_feedback_preferences_with_principles500.json


In [ ]:
# Without principles
use_principles = False
out_json = f"ai_feedback_preferences_without_principles{max_examples}.json"
generate_dataset(use_principles,out_json)



[data] Subset size: 500
[principles] Skipped (use_principles=False)
[model] Loaded on cpu


Labeling: 100%|██████████████████████████████████████████████████████████████████████| 500/500 [37:24<00:00,  4.49s/it]

[done] Saved 500 → ai_feedback_preferences_without_principles500.json
